In [ ]:
import sys
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
PARENT = HERE.parent.parent.parent  # server/scripts
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))
MAP_PATH = PARENT / "server/map/grid"
MAP_PATH.mkdir(parents=True, exist_ok=True)
RANKED_BUCKET_PATH = PARENT / "server/out/places_ranked"
DF_RANKED = pd.read_csv(RANKED_BUCKET_PATH / "places_scored_level_1.csv")
DF_RANKED.drop(columns=['wilson_0', 'normal_0', 'wilson_2', 'normal_2', 'h3_res9'])

In [ ]:
import numpy as np

TYPE_COL   = 'cuisineType'
SCORE_COL  = 'wilson_1'
TOP_SLICE  = 0.25   # fraction used to measure over-representation
LAMBDA     = 0.35   # penalty strength (0 = no penalty, 1 = strong)

df_bias = DF_RANKED.copy()
df_bias[TYPE_COL] = df_bias[TYPE_COL].fillna('(missing)')

# 1. Global share of each cuisine across all restaurants
global_share = df_bias[TYPE_COL].value_counts(normalize=True).rename('global_share')

# 2. Share of each cuisine inside the top slice
k = max(int(len(df_bias) * TOP_SLICE), 1)
top_slice = df_bias.nlargest(k, SCORE_COL)
top_share = top_slice[TYPE_COL].value_counts(normalize=True).rename('top_share')

# 3. Representation ratio  RR = top_share / global_share
bias = pd.concat([global_share, top_share], axis=1).fillna(0)
bias['repr_ratio'] = bias['top_share'] / bias['global_share'].replace(0, np.nan)
bias['repr_ratio'] = bias['repr_ratio'].fillna(0)

# 4. Soft penalty  exp(-λ · max(log(RR), 0))
#    RR ≤ 1  →  penalty = 1.0  (no effect)
#    RR > 1  →  penalty < 1.0  (gentle downweight)
bias['soft_penalty'] = np.exp(-LAMBDA * np.maximum(np.log(bias['repr_ratio'].replace(0, np.nan)).fillna(0), 0))

bias.sort_values('repr_ratio', ascending=False)